# 03 — Post-Level Common Features

**Input**: `merged_post_level.csv` (patched, with `amp_prob_full`, `amp_prob_behavioral`, `is_sockpuppet`)

**Output**: `post_features.csv` — one row per post, with language-agnostic features computable by identical code on both platforms.

## Why post-level

User-level aggregation is broken for this dataset: 93.7% of TikTok users posted exactly once. Lindsey's per-user features (mean/max/std/sum across a user's posts) collapse to a single observation × 4 redundant stats. Working at post level is honest given the data: each post is the natural unit of observation, both labels are post-level, and we have ~25,000 posts vs ~18,800 "users" most of which are pseudo-users.

## Feature set

**6 text-structure features** (language-agnostic by construction):
- `n_chars`         — total character count
- `n_words`         — whitespace-token count *(caveat: Thai tokenizes weakly with whitespace)*
- `n_emoji`         — Unicode emoji count
- `n_punct`         — `!?.,` count
- `emoji_density`   — emoji / max(words, 1), capped at 1
- `punct_density`   — punct / max(chars, 1)

**3 anomaly flags** (Lindsey's original rule logic, kept):
- `is_short`        — n_chars < 5
- `is_long`         — n_chars > 500
- `is_emoji_heavy`  — n_emoji >= 4 AND n_words <= 4

**2 repetition signals**:
- `max_token_repeat_ratio` — max-repeated-token fraction
- `has_repeated_chars`     — any character repeated >= 5x consecutively

**3 temporal features**:
- `hour_of_day`     — 0-23
- `day_of_week`     — 0=Mon
- `is_late_night`   — hour in [0, 5]

**Total: 14 features.** All computed by identical code on both platforms.

## What is NOT included and why

- **`likes_received`, `replies_received`** — construct mismatch (TikTok algorithmic likes vs Pantip member reactions).
- **Inter-post temporal features** (avg/std time between posts) — user-level only, broken by single-post users.
- **All Filipino-lexicon scores** — by design, this is the transfer test.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

IN_PATH  = 'merged_post_level.csv'
OUT_PATH = 'post_features.csv'

df = pd.read_csv(IN_PATH)
df['text']     = df['text'].fillna('').astype(str)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
print(f'Loaded {len(df):,} posts')
print(df['source'].value_counts())

## Single feature-extraction function

Same code path for every row, regardless of `source`. This is the methodological commitment that makes the cross-platform claim defensible.

In [ ]:
EMOJI_PAT = re.compile(
    r'[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]|'
    r'[\U0001F680-\U0001F6FF]|[\U0001F1E0-\U0001F1FF]|'
    r'[\U00002700-\U000027BF]|[\U0001F900-\U0001F9FF]'
)
REPEAT_CHAR_PAT = re.compile(r'(.)\1{4,}')

def extract_post_features(text, dt):
    out = {}
    n_chars = len(text)
    words   = text.split()
    n_words = len(words)
    n_emoji = len(EMOJI_PAT.findall(text))
    n_punct = sum(1 for ch in text if ch in '!?.,')

    out['n_chars']        = n_chars
    out['n_words']        = n_words
    out['n_emoji']        = n_emoji
    out['n_punct']        = n_punct
    out['emoji_density']  = min(n_emoji / max(n_words, 1), 1.0)
    out['punct_density']  = n_punct / max(n_chars, 1)

    out['is_short']       = int(n_chars < 5)
    out['is_long']        = int(n_chars > 500)
    out['is_emoji_heavy'] = int(n_emoji >= 4 and n_words <= 4)

    if n_words > 1:
        top = Counter([w.lower() for w in words]).most_common(1)[0][1]
        out['max_token_repeat_ratio'] = (top - 1) / n_words
    else:
        out['max_token_repeat_ratio'] = 0.0
    out['has_repeated_chars'] = int(bool(REPEAT_CHAR_PAT.search(text)))

    if pd.notna(dt):
        out['hour_of_day']   = dt.hour
        out['day_of_week']   = dt.dayofweek
        out['is_late_night'] = int(0 <= dt.hour <= 5)
    else:
        out['hour_of_day']   = -1
        out['day_of_week']   = -1
        out['is_late_night'] = 0
    return out

samples = [
    ('BBM SARA UNITY!!! flags', pd.Timestamp('2022-05-09 02:30')),
    ('agree definitely yes', pd.Timestamp('2019-03-24 14:00')),
    ('yessssss', pd.Timestamp('2022-05-09 18:00')),
    ('', pd.NaT),
]
for txt, dt in samples:
    f = extract_post_features(txt, dt)
    print(f'"{txt[:30]}"')
    print(f'  {f}')

## Apply to all posts

In [ ]:
print(f'Extracting features for {len(df):,} posts...')
feats = df.apply(lambda r: extract_post_features(r['text'], r['datetime']), axis=1)
feat_df = pd.DataFrame(list(feats), index=df.index)

keep = ['source', 'post_id', 'user_id', 'container_id',
        'amp_prob_full', 'amp_prob_behavioral', 'is_sockpuppet']
out = pd.concat([df[keep], feat_df], axis=1)
print(f'Result shape: {out.shape}')
print(f'Feature columns: {list(feat_df.columns)}')

## Cross-platform distribution diagnostic

For each feature, compare its distribution across the negative class of each platform. If a feature has very different scale between TikTok negatives and Pantip negatives, it's encoding platform identity rather than amplifier signal.

Negative class definitions:
- TikTok negative: `amp_prob_behavioral < 0.5` (the headline label)
- Pantip negative: `is_sockpuppet == 0`

In [ ]:
feat_cols = list(feat_df.columns)
neg_tt = out[(out['source']=='tiktok') & (out['amp_prob_behavioral']<0.5)]
neg_pt = out[(out['source']=='pantip') & (out['is_sockpuppet']==0)]

diag = pd.DataFrame({
    'tt_neg_mean': neg_tt[feat_cols].mean(),
    'pt_neg_mean': neg_pt[feat_cols].mean(),
    'tt_neg_std':  neg_tt[feat_cols].std(),
    'pt_neg_std':  neg_pt[feat_cols].std(),
})
diag['ratio'] = (diag['tt_neg_mean'] / diag['pt_neg_mean'].replace(0, np.nan)).abs()
diag['flag']  = (diag['ratio'] > 2) | (diag['ratio'] < 0.5)
print('NEGATIVE-CLASS DISTRIBUTION DIAGNOSTIC')
print('='*80)
print(diag.round(3).to_string())
print()
print(f'Flagged (>2x or <0.5x scale gap): {diag["flag"].sum()} of {len(diag)} features')

## Positive vs negative discrimination check (per platform)

Sanity: for each feature, does it separate positives from negatives *within* each platform? If a feature has zero within-platform discrimination, it can't carry transfer signal.

In [ ]:
def pos_neg_gap(sub, label_col, positive_pred):
    pos = sub[positive_pred(sub[label_col])]
    neg = sub[~positive_pred(sub[label_col])]
    return (pos[feat_cols].mean() - neg[feat_cols].mean()).round(3)

tt = out[out['source']=='tiktok']
pt = out[out['source']=='pantip']

disc = pd.DataFrame({
    'tt_behavioral_gap': pos_neg_gap(tt, 'amp_prob_behavioral', lambda s: s>=0.5),
    'tt_full_gap':       pos_neg_gap(tt, 'amp_prob_full',       lambda s: s>=0.5),
    'pt_sockpuppet_gap': pos_neg_gap(pt, 'is_sockpuppet',       lambda s: s==1),
})
print('Mean(positive) - Mean(negative) per feature, per label:')
print(disc.to_string())
print()
print('Interpretation:')
print('  Same-sign across columns = feature direction transfers (good)')
print('  Opposite-sign = feature flips meaning across platforms (bad)')
print('  Near-zero on Pantip = feature has no Pantip discrimination (cannot transfer)')

## Export

In [ ]:
out.to_csv(OUT_PATH, index=False)
print(f'Saved -> {OUT_PATH}')
print(f'  Rows: {len(out):,}')
print(f'  Cols: {out.shape[1]} (7 id/label + 14 features)')
print()
print('Next: 04_zero_shot_transfer.ipynb')
print('  - Train on TikTok (label = amp_prob_behavioral >= 0.5)')
print('  - Test on Pantip  (label = is_sockpuppet)')
print('  - Ablation: train on amp_prob_full, show the lexicon-bound label degrades transfer.')